# 01 – Data exploration and CSP + LDA baseline

Euclidean baseline for 4-class motor imagery on BCI Competition IV 2a.
Data pipeline and design choices: `src/riemannian_eeg/data.py` and `docs/design_decisions.md`.

In [ ]:
from riemannian_eeg.data import load_train_test

X_train, y_train, X_test, y_test = load_train_test(subject=1)
print(X_train.shape, X_test.shape)

## CSP + LDA baseline

CSP + LDA is the classic pipeline for motor imagery, so it's our reference point. It uses
the same information as the Riemannian methods (covariances between electrodes) but treats
them as if they lived in a flat, Euclidean space, so any accuracy gap with MDM, tangent
space or SPDNet comes from the geometry.

### Spatial filters as virtual electrodes

Each trial is a matrix $X \in \mathbb{R}^{22 \times T}$ (electrodes × time samples). A spatial
filter $w \in \mathbb{R}^{22}$ combines the electrodes into one new signal, a "virtual electrode":

$$
s(t) = w^\top x(t) = \sum_{c=1}^{22} w_c \, x_c(t).
$$

Any signal can be seen as a sum of oscillations at different frequencies, plus a constant
offset that doesn't oscillate at all (0 Hz): that offset is the signal's mean. The 8-30 Hz
band-pass filter only keeps oscillations between 8 and 30 Hz, so it removes the offset and
leaves the signal oscillating around zero (zero mean).

**Trial covariance.** Each trial has its own $22 \times 22$ covariance matrix, computed from its
$T$ time samples. Since the signal has zero mean,

$$
C = \frac{1}{T} X X^\top, \qquad C_{cd} = \frac{1}{T} \sum_t x_c(t)\, x_d(t).
$$

Its diagonal $C_{cc}$ is the power of each electrode, and the off-diagonal entries $C_{cd}$
measure how much two electrodes vary together.

**Power of a virtual electrode.** The power of the virtual electrode $s(t) = w^\top x(t)$ is the
average of its squared values. It depends on $w$, because the weights decide how the electrodes
are combined, and it can be computed directly from $C$. With two electrodes, for instance:

$$
s(t)^2 = w_1^2\, x_1(t)^2 + 2\, w_1 w_2\, x_1(t)\, x_2(t) + w_2^2\, x_2(t)^2,
$$

and averaging over time (the weights are constants, so they come out of the average):

$$
\operatorname{Var}(s) = w_1^2\, C_{11} + 2\, w_1 w_2\, C_{12} + w_2^2\, C_{22} = w^\top C \, w.
$$

The same holds for 22 electrodes, just with more terms. So once we have $C$, we can get the
power of *any* combination of electrodes without going back to the raw signal: $C$ describes
the trial, and $w$ says how the electrodes are combined.

### The CSP objective

Applying the same filter $w$ to the trials of each class, $w^\top C_1 w$ is the power of the
virtual electrode for class 1 and $w^\top C_2 w$ its power for class 2, where $C_1$ and $C_2$ are
the average covariances of each class. CSP looks for the filter whose power is as large as
possible for class 1 and as small as possible for class 2, i.e. it maximizes their ratio:

$$
\max_w \; J(w) = \frac{w^\top C_1 w}{w^\top C_2 w}.
$$

**Only the direction of $w$ matters.** Rescaling $w$ by any $a \neq 0$ multiplies both powers by
$a^2$, which cancels out: $J(aw) = J(w)$. So we're free to pick the scale of $w$, and we choose
the one that makes the denominator equal to 1. The problem becomes a constrained one:

$$
\max_w \; w^\top C_1 w \quad \text{subject to} \quad w^\top C_2 w = 1.
$$

**Lagrange multipliers.** To maximize a function under a constraint, we build the Lagrangian,
which adds the constraint (set equal to zero) multiplied by a new variable $\lambda$:

$$
L(w, \lambda) = w^\top C_1 w - \lambda \left(w^\top C_2 w - 1\right).
$$

At the optimal point, the gradient of $L$ with respect to $w$ must be zero (otherwise we could
still move along the constraint and increase the power ratio). Since $C_1$ and $C_2$ are
symmetric, the derivative of a quadratic form is $\frac{\partial}{\partial w}(w^\top A w) = 2Aw$
(the matrix version of $\frac{d}{dx}(a x^2) = 2ax$), so:

$$
\frac{\partial L}{\partial w} = 2\,C_1 w - 2\lambda\, C_2 w = 0
\quad\Longrightarrow\quad
C_1 w = \lambda \, C_2 w.
$$

This is a **generalized eigenvalue problem**: the solutions $w$ are the generalized eigenvectors,
each with its eigenvalue $\lambda$.

**Meaning of $\lambda$.** Multiplying both sides by $w^\top$ on the left:

$$
w^\top C_1 w = \lambda \, w^\top C_2 w
\quad\Longrightarrow\quad
\lambda = \frac{w^\top C_1 w}{w^\top C_2 w} = J(w).
$$

So each eigenvalue is exactly the power ratio of its filter. Filters with the largest eigenvalues
have high power in class 1 and low in class 2, and those with the smallest eigenvalues the
opposite. We keep a few from each end and drop the middle ones, which don't discriminate.

**From each trial to a few features.** Solving the eigenvalue problem gives a few filters
$w_1, \dots, w_k$ (e.g. $k = 6$, taken from both ends of the spectrum). For each trial, we compute
the power of every virtual electrode with that trial's covariance $C$, and take its log:

$$
f_j = \log\!\left(w_j^\top C \, w_j\right), \quad j = 1, \dots, k.
$$

So a trial that was a $22 \times 1001$ matrix ends up summarized by just $k$ numbers, the features
that LDA receives. The log is there because powers are always positive and very skewed (most are
small, a few are huge); taking the log spreads them out and makes each class roughly Gaussian,
which is what LDA assumes.

**Where CSP is Euclidean.** To solve the problem we need $C_1$ and $C_2$, the average covariance of
each class. CSP computes them as plain arithmetic means: add up the covariance matrices of all the
trials of a class and divide by their number, as if SPD matrices lived in a flat space. But
covariances measure power, which is always positive and is naturally compared through ratios
rather than differences. The arithmetic mean ignores this: it is pulled towards the trials with
the largest power, and the resulting average can look more spread out (larger determinant) than
the matrices it averages, the so-called swelling effect. The Riemannian methods instead average
along the curved SPD manifold (the Fréchet mean), which respects these ratios and avoids the
distortion. This averaging step is exactly what changes between CSP and the Riemannian methods,
which is why CSP is the right baseline to compare against.

### Physiological intuition

The motor cortex is organized by body part: the area controlling each part of the body lies at a
different position along it, and each hemisphere controls the opposite side of the body. When a
movement is imagined, the mu/beta power over the corresponding area drops (Event-Related
Desynchronization, ERD), and over other areas it can increase (Event-Related Synchronization, ERS).
Over the electrodes we have, this gives a characteristic pattern for each class:

| Imagined movement | Motor area involved | Where the power drops (ERD) |
|---|---|---|
| Left hand | Hand area, right hemisphere | Around C4 |
| Right hand | Hand area, left hemisphere | Around C3 |
| Feet | Foot area, on the midline between hemispheres | Around Cz, weaker because the area lies deep |
| Tongue | Face/tongue area, lateral and lower | Less focal; often ERS over the hand areas (C3, C4) |

So the hand classes are told apart by the left-right contrast between C3 and C4, while feet and
tongue are recognized by a central ERD (feet) or by the hand areas becoming *more* active
(ERS) instead of less. CSP isn't told any of this: it only receives the trials and their labels.
But since its filters look for directions where power differs between classes, they usually end
up weighting exactly these electrodes. Because it keeps filters from both ends of the eigenvalue
spectrum, it captures both power drops (ERD) and power increases (ERS).

### From 2 to 4 classes

The CSP objective compares two classes ($C_1$ vs $C_2$), so with four classes it has to be
extended. We compare two standard options.

**One-vs-Rest (OvR).** We train 4 independent binary models, one per class. In the model for
class $k$, $C_1$ is the average covariance of class $k$ and $C_2$ the average covariance of the
other three classes pooled together. Each model learns its own CSP filters (the directions that
best separate *its* class from the rest) and its own LDA. To classify a new trial, the 4 models
give a score and we pick the class whose model is most confident.

**Multiclass CSP by joint diagonalization (MNE).** In the two-class case, the generalized
eigenvectors make $W^\top C_1 W$ and $W^\top C_2 W$ both diagonal: in the filtered space, each
virtual electrode has its own power, with no cross-terms between them. With four classes there is
in general no single $W$ that diagonalizes the four $C_k$ exactly, so MNE finds the $W$ that
makes them *as diagonal as possible* at the same time (approximate joint diagonalization). This
gives one shared set of filters for all classes, and MNE keeps the ones that carry the most
information about the class label (Grosse-Wentrup & Buss, 2008).

The trade-off: OvR gives each class filters tailored to it, but each model only sees a "one
class vs everything" contrast. Joint diagonalization uses a single set of filters fitted on all
four classes at once. We don't consider One-vs-One (one model per pair of classes), since it
would need 6 models, each trained on only half the data.

The variant is chosen by cross-validation on the training session only, so the test session
stays untouched until the final evaluation.

### LDA

**LDA (Linear Discriminant Analysis)** separates the CSP features with linear boundaries.
It is simple, fast and works well with few training trials.

*Refs: Ramoser, Müller-Gerking & Pfurtscheller (2000), IEEE Trans. Rehabil. Eng. 8(4);
Blankertz et al. (2008), IEEE Signal Process. Mag. 25(1);
Grosse-Wentrup & Buss (2008), "Multiclass common spatial patterns and information theoretic
feature extraction", IEEE Trans. Biomed. Eng. 55(8).*

In [ ]:
from riemannian_eeg.data import load_train_test

X_train, y_train, X_test, y_test = load_train_test(subject=1)
print(X_train.shape, X_test.shape)